<img src="https://raw.githubusercontent.com/drdave-teaching/OPIM5641-notebooks/main/_banners/opim5641_banner.svg" width="100%" alt="OPIM 5641 banner"/>

**Dr. Dave Wanik - Operations and Information Management - University of Connecticut**

---

# Studio 5 — Your Own Portfolio: Real Data and Business Rules

In Studio 4 you solved Ms. Womack's model and ran it on stocks someone else picked. Tonight it is your turn:

1. **Part 1 · Download your own stocks** from Yahoo Finance and turn the prices into monthly returns.
2. **Part 2 · Ms. Womack's model, plus business rules.** The yes/no decisions from Module 5: if you invest in a
   stock at all, put in at least 2%; never more than 10% in any one stock; hold at least one stock from the
   sectors you choose.
3. **Part 3 · What the rules cost.** The efficient frontier with and without the rules, side by side.

All of the code is written. You choose the stocks and the rules. When you see **Predict first**, write down your
guess before you run the next cell.

## Import modules (download Pyomo, ipopt, Bonmin and yfinance)

Run this first, every time. It installs Pyomo with two solvers: **ipopt** (nonlinear, from Module 4) and
**Bonmin** (nonlinear *and* yes/no variables, new tonight). It also installs `yfinance`. About a minute.

In [ ]:
# install Pyomo, the solvers and yfinance in Colab (idaes-pse brings ipopt and Bonmin)
import sys
import os

if 'google.colab' in sys.modules:
    !pip install -q idaes-pse --pre yfinance
    !idaes get-extensions --to ./bin
    os.environ['PATH'] += ':bin'

%matplotlib inline
from pylab import *
import numpy as np
import pandas as pd
from pyomo.environ import *
from IPython.display import display, Math

import logging
logging.getLogger('pyomo.core').setLevel(logging.ERROR)

# where the setup cell put the solvers
ipopt_path = '/content/bin/ipopt'
bonmin_path = '/content/bin/bonmin'

# Part 1 · Download your own stocks

### Step 1 · Pick your stocks and give each one a sector

Edit the dictionary below: **one line per stock**, the ticker on the left and its sector on the right. It starts
with 20 well-known companies across 8 sectors. Add, remove or swap any of them (for example `'NFLX':
'Communication Services',`). Keep at least **12 stocks**: tonight's rules need room to spread the money out.

Not sure of a ticker? Search the company name on finance.yahoo.com. Sectors are up to you, but use the same
spelling for stocks in the same sector.

In [ ]:
# your stocks: ticker -> sector (one line per stock)
my_stocks = {
    'AAPL': 'Technology',
    'MSFT': 'Technology',
    'NVDA': 'Technology',
    'GOOGL': 'Communication Services',
    'META': 'Communication Services',
    'JPM': 'Financials',
    'GS': 'Financials',
    'V': 'Financials',
    'JNJ': 'Health Care',
    'LLY': 'Health Care',
    'UNH': 'Health Care',
    'XOM': 'Energy',
    'CVX': 'Energy',
    'KO': 'Consumer Staples',
    'WMT': 'Consumer Staples',
    'COST': 'Consumer Staples',
    'AMZN': 'Consumer Discretionary',
    'HD': 'Consumer Discretionary',
    'CAT': 'Industrials',
    'LMT': 'Industrials',
}

# the five years of monthly data we want (first day of the first and last month)
start = '2021-09-01'
end = '2026-10-01'

print(len(my_stocks), "stocks in", len(set(my_stocks.values())), "sectors")

### Step 2 · Download the prices

This asks Yahoo Finance for the **month-end, adjusted** price of each stock (adjusted for splits and dividends:
remember the NVIDIA split trap from Studio 4.3). It downloads one stock at a time and tells you if any fail.

Yahoo sometimes blocks Colab. If the download fails, the cell falls back to a copy of real prices we saved for
class, so you can keep going either way.

In [ ]:
import yfinance as yf

prices = pd.DataFrame()
failed = []
for ticker in my_stocks:
    try:
        data = yf.download(ticker, start=start, end=end, interval='1mo', auto_adjust=True, progress=False)
        if len(data) > 12:
            prices[ticker] = data['Close'].squeeze()
        else:
            failed.append(ticker)
    except Exception as problem:
        failed.append(ticker)

if len(failed) > 0:
    print("These did not download:", failed)

# the fallback: real month-end prices saved for class (55 stocks, Sep 2021 - Sep 2026)
if len(prices.columns) < 12:
    print("Yahoo did not give us enough stocks, so we are using the saved class copy instead.")
    saved = pd.read_csv('https://raw.githubusercontent.com/drdave-teaching/OPIM5641-notebooks/main/studio4/data/sector_prices_monthly.csv', index_col=0)
    lookup = pd.read_csv('https://raw.githubusercontent.com/drdave-teaching/OPIM5641-notebooks/main/studio4/data/sector_lookup.csv')
    prices = pd.DataFrame()
    for ticker in my_stocks:
        if ticker in saved.columns:
            prices[ticker] = saved[ticker]
    if len(prices.columns) < 12:
        prices = saved[list(lookup['ticker'])]
        my_stocks = {}
        for i in range(len(lookup)):
            my_stocks[lookup.loc[i, 'ticker']] = lookup.loc[i, 'sector']

print(prices.shape[0], "months of prices for", prices.shape[1], "stocks")
prices.tail()

### Step 3 · Prices into monthly returns, then check them

Same three steps as Studio 4.3: returns, then the checks (how many months, anything missing, any impossible moves).

$$r_t = \frac{P_t}{P_{t-1}} - 1$$

In [ ]:
returns = prices.pct_change()
returns = returns.dropna(how='all')

# drop any stock that is missing months (a company that did not exist yet, a ticker that changed)
complete = []
for ticker in returns.columns:
    if returns[ticker].isna().sum() == 0:
        complete.append(ticker)
    else:
        print("dropping", ticker, "- missing", returns[ticker].isna().sum(), "months")
returns = returns[complete]

print("months:", len(returns))
print("stocks:", len(returns.columns))
print("missing values left:", returns.isna().sum().sum())

moves = pd.DataFrame({'worst month': returns.min(), 'best month': returns.max(), 'average month': returns.mean()})
moves.sort_values('worst month').round(3)

**Look before you model.** Any month worse than -60%? That is usually a split or a data error, not a crash.
Which stock had the best average month? Write it down: we will see whether the model goes all-in on it.

In [ ]:
# save your returns so you can use them again (Files panel on the left, or File > Download)
returns.to_csv('my_monthly_returns.csv')

df_return = returns.mean()
df_cov = returns.cov()
stocks = list(returns.columns)
print("saved my_monthly_returns.csv:", len(stocks), "stocks")

# Part 2 · Ms. Womack's model, plus business rules

### The model you already know (Studio 4.2)

$$
\begin{aligned}
\max \quad & \sum_k \bar r_k \, x_k && \text{(average return)} \\
\text{subject to} \quad & \sum_k x_k = 1 && \text{(invest all of the money)} \\
& \sum_i \sum_j x_i \, x_j \, \text{Cov}(i,j) \le \text{risk limit} && \text{(the risk)} \\
& 0 \le x_k \le 1
\end{aligned}
$$

### New tonight: a yes/no decision for every stock

The business rules are about **whether** you hold a stock, not just how much. So each stock gets a second
decision variable, a **binary** (yes/no) one, exactly like project selection in M5.1:

$$y_k = \begin{cases} 1 & \text{we hold stock } k \\ 0 & \text{we do not} \end{cases}$$

Then each rule is a constraint that **links** $x_k$ (how much) to $y_k$ (whether):

$$
\begin{array}{lcl}
\textbf{Business rule} & \qquad\textbf{Constraint}\qquad & \textbf{What it does} \\[8pt]
\text{Never more than }10\%\text{ in one stock} & x_k \le 0.10\, y_k & \text{if } y_k = 0 \text{, the share must be 0; if } y_k = 1 \text{, up to }10\% \\[10pt]
\text{If you invest at all, at least }2\% & x_k \ge 0.02\, y_k & \text{if } y_k = 1 \text{, at least }2\%\text{ (no tiny }0.3\%\text{ positions)} \\[10pt]
\text{At least one stock from a sector} & \displaystyle\sum_{k \in \text{sector}} y_k \ge 1 & \text{counts the yes/no picks in that sector} \\[14pt]
\text{(optional) At most } K \text{ stocks} & \displaystyle\sum_{k} y_k \le K & \text{caps how many stocks you hold}
\end{array}
$$

Read the first two rows together. Here is every case for one stock:

$$
\begin{array}{c|c|c|l}
y_k & \quad x_k \le 0.10\, y_k \quad & \quad x_k \ge 0.02\, y_k \quad & \text{so } x_k \text{ is...} \\[4pt] \hline
0 & x_k \le 0 & x_k \ge 0 & \text{exactly 0: we do not hold it} \\[4pt]
1 & x_k \le 0.10 & x_k \ge 0.02 & \text{between }2\%\text{ and }10\%
\end{array}
$$

A model with yes/no variables **and** a nonlinear risk is a **mixed-integer nonlinear** program. ipopt cannot handle
the yes/no part, so tonight we use **Bonmin**, the solver from Dave's integer version of Ms. Womack.

### Step 4 · Choose your rules

Edit the dictionary. `None` switches a rule off.

In [ ]:
rules = {
    'min_share': 0.02,                                            # if you invest at all, at least 2%
    'max_share': 0.10,                                            # never more than 10% in one stock
    'must_have_sectors': ['Technology', 'Health Care', 'Energy'],  # at least one stock from each
    'max_stocks': None,                                           # e.g. 12, or None for no limit
}

# a quick sanity check: can the rules even be met with your stocks?
if rules['max_share'] is not None and len(stocks) * rules['max_share'] < 1:
    print("Not enough stocks for a %d%% cap: add stocks or raise max_share." % (100 * rules['max_share']))
else:
    print("OK:", len(stocks), "stocks, so a cap of", rules['max_share'], "can still invest all of the money")

### Step 5 · The model, with a switch for the rules

`solve_portfolio(risk_limit, use_rules)` is Studio 4.2's model plus the yes/no variables. With `use_rules=False`
it is exactly Ms. Womack (solved with ipopt); with `use_rules=True` it adds every rule in the dictionary
(solved with Bonmin).

In [ ]:
def solve_portfolio(risk_limit, use_rules):
    model = ConcreteModel()

    # how much of the money goes in each stock
    model.x = Var(stocks, domain=NonNegativeReals, bounds=(0, 1))
    for s in stocks:
        model.x[s] = 1 / len(stocks)

    # objective: the average return of the portfolio
    obj_expr = 0
    for s in stocks:
        obj_expr += df_return[s] * model.x[s]
    model.portfolio_return = Objective(expr=obj_expr, sense=maximize)

    model.constraints = ConstraintList()

    # invest all of the money
    budget = 0
    for s in stocks:
        budget += model.x[s]
    model.constraints.add(budget == 1)

    # the risk: the weighted covariance matrix, summed
    risk_expr = 0
    for i in stocks:
        for j in stocks:
            risk_expr += model.x[i] * df_cov.loc[i, j] * model.x[j]
    model.constraints.add(risk_expr <= risk_limit)

    if use_rules:
        # the yes/no decision: do we hold stock s?
        model.y = Var(stocks, domain=Binary)
        for s in stocks:
            model.y[s] = 1
        for s in stocks:
            if rules['max_share'] is not None:
                model.constraints.add(model.x[s] <= rules['max_share'] * model.y[s])
            else:
                model.constraints.add(model.x[s] <= model.y[s])
            if rules['min_share'] is not None:
                model.constraints.add(model.x[s] >= rules['min_share'] * model.y[s])
        for sector in rules['must_have_sectors']:
            picks = 0
            for s in stocks:
                if my_stocks.get(s) == sector:
                    picks += model.y[s]
            model.constraints.add(picks >= 1)
        if rules['max_stocks'] is not None:
            held = 0
            for s in stocks:
                held += model.y[s]
            model.constraints.add(held <= rules['max_stocks'])
        results = SolverFactory('bonmin', executable=bonmin_path).solve(model)
    else:
        results = SolverFactory('ipopt', executable=ipopt_path).solve(model)

    if results.solver.termination_condition != TerminationCondition.optimal:
        return None
    weights = {}
    for s in stocks:
        weights[s] = model.x[s]()
    return {'return': model.portfolio_return(), 'risk': value(risk_expr), 'weights': weights}


print("solve_portfolio is ready")

### Step 6 · One risk limit, with and without the rules

We use the same risk limit for both: about the risk of an even split of your stocks (so it is always reachable).

**Predict first.** Will the rules make the best return go **up**, **down**, or stay the **same**? Write it down.

In [ ]:
even = {}
for s in stocks:
    even[s] = 1 / len(stocks)
even_risk = 0
for i in stocks:
    for j in stocks:
        even_risk += even[i] * even[j] * df_cov.loc[i, j]
risk_limit = round(even_risk, 4)
print("risk limit:", risk_limit, "(the risk of splitting your money evenly across all your stocks)")

free = solve_portfolio(risk_limit, use_rules=False)
ruled = solve_portfolio(risk_limit, use_rules=True)

held_free = 0
held_ruled = 0
for s in stocks:
    if free['weights'][s] > 0.001:
        held_free += 1
    if ruled['weights'][s] > 0.001:
        held_ruled += 1
# what each portfolio would earn on $10,000 in an average month (based on the past five years)
invested = 10000
free_dollars = invested * free['return']
ruled_dollars = invested * ruled['return']
cost_points = 100 * (free['return'] - ruled['return'])
cost_dollars = free_dollars - ruled_dollars

print()
print("Same risk limit for both. On $10,000, in an average month:")
print()
print("                     average return    you would earn    stocks held")
free_text = "$%.0f a month" % free_dollars
ruled_text = "$%.0f a month" % ruled_dollars
print("  no rules           %5.2f%% a month    %-16s  %d" % (100 * free['return'], free_text, held_free))
print("  with your rules    %5.2f%% a month    %-16s  %d" % (100 * ruled['return'], ruled_text, held_ruled))
print()
print("Following the rules costs %.2f percentage points a month:" % cost_points)
print("about $%.0f less a month on $10,000, or about $%.0f less a year." % (cost_dollars, 12 * cost_dollars))

In [ ]:
# what each portfolio holds, side by side
holdings = pd.DataFrame({'no rules': free['weights'], 'with rules': ruled['weights']})
holdings = holdings[(holdings['no rules'] > 0.001) | (holdings['with rules'] > 0.001)]
holdings = holdings.sort_values('no rules', ascending=False)
holdings.plot(kind='bar', figsize=(12, 4), color=['gray', 'tab:orange'])
axhline(rules['max_share'], color='black', linestyle='--', linewidth=1, label='10% cap')
axhline(rules['min_share'], color='black', linestyle=':', linewidth=1, label='2% floor')
ylabel('Share of the money')
title('What to hold at risk limit %.4f' % risk_limit)
legend()
show()

**Read the bars.** Without rules the model piles into a few stocks with the best past averages. With rules every
position sits between the dotted 2% floor and the dashed 10% cap, so the money is spread over at least 10 stocks.
Adding a constraint can never make the best return go up (the rule-free answer was always available without the
rules), so the rules can only cost you return. What they buy you is a portfolio you could actually defend to Ms.
Womack.

# Part 3 · What the rules cost: two frontiers

Solve both models at many risk limits and draw both efficient frontiers. The gap between the curves is the price of
the business rules, at every level of risk. (The rules version takes a little longer: Bonmin tries yes/no
combinations.)

In [ ]:
# risk limits from just above the safest reachable mix up to the riskiest single stock
top = df_cov.values.diagonal().max()
levels = np.linspace(even_risk * 0.5, top, 15)

free_curve = {}
free_weights = {}
ruled_curve = {}
for r in levels:
    a = solve_portfolio(r, use_rules=False)
    if a is not None:
        free_curve[r] = a['return']
        free_weights[r] = a['weights']
    b = solve_portfolio(r, use_rules=True)
    if b is not None:
        ruled_curve[r] = b['return']

# where does the rules line go flat? solve once with the highest risk limit, then read off how much risk it used
ruled_best = solve_portfolio(top, use_rules=True)
flat_risk = ruled_best['risk']
print("with your rules, the best return is %.2f%% a month, and it only needs a risk of %.4f"
      % (100 * ruled_best['return'], flat_risk))
print("past that point, allowing more risk buys nothing: your rules are what hold the return back")

# a closer look at the rules line, from the lowest risk limit to just past the flat point
zoom_levels = np.linspace(levels[0], flat_risk * 1.3, 15)
ruled_zoom = {}
ruled_weights = {}
for r in zoom_levels:
    b = solve_portfolio(r, use_rules=True)
    if b is not None:
        ruled_zoom[r] = b['return']
        ruled_weights[r] = b['weights']


def find_corner(curve):
    # the top-left corner: scale both axes from 0 to 1, draw a straight line from the first point to the last,
    # and take the point that sits farthest above that line. Past it, extra risk buys less and less return.
    risks = list(curve.keys())
    returns = list(curve.values())
    if returns[-1] == returns[0]:
        return risks[0]
    best_risk = risks[0]
    best_gap = -1
    for i in range(len(risks)):
        x = (risks[i] - risks[0]) / (risks[-1] - risks[0])
        y = (returns[i] - returns[0]) / (returns[-1] - returns[0])
        if y - x > best_gap:
            best_gap = y - x
            best_risk = risks[i]
    return best_risk


corner_risk = find_corner(ruled_zoom)
corner_return = ruled_zoom[corner_risk]
print("top-left corner of your orange line: risk limit %.4f, return %.2f%% a month" % (corner_risk, 100 * corner_return))

fig, panels = subplots(1, 2, figsize=(15, 5.5))

left = panels[0]
left.plot(list(free_curve.keys()), list(free_curve.values()), marker='o', color='gray', label='no rules')
# the orange line on the left uses both sets of points, so the zoomed-in corner sits on it
ruled_all = dict(ruled_curve)
ruled_all.update(ruled_zoom)
ruled_risks = sorted(ruled_all.keys())
ruled_returns = []
for r in ruled_risks:
    ruled_returns.append(ruled_all[r])
left.plot(ruled_risks, ruled_returns, marker='o', color='tab:orange', label='with your rules')
left.axvline(flat_risk, color='tab:orange', linestyle=':', label='rules line goes flat')
left.plot(corner_risk, corner_return, marker='*', markersize=18, color='black', linestyle='none', label='top-left corner')
left.set_xlabel('Risk limit (variance of the monthly return)')
left.set_ylabel('Best average monthly return')
left.set_title('Both frontiers: the gap is the price of the rules')
left.legend()

right = panels[1]
right.plot(list(ruled_zoom.keys()), list(ruled_zoom.values()), marker='o', color='tab:orange', label='with your rules')
right.axvline(flat_risk, color='tab:orange', linestyle=':', label='flat from here on')
right.plot(corner_risk, corner_return, marker='*', markersize=18, color='black', linestyle='none', label='top-left corner')
right.set_xlabel('Risk limit (variance of the monthly return)')
right.set_ylabel('Best average monthly return')
right.set_title('Zoomed in: the rules frontier on its own')
right.legend()

show()

**Read the two panels.** On the left, the gray line keeps climbing: with no rules, more risk lets the model pile into
the few stocks with the best past averages. The orange line goes flat almost right away. That is the 10% cap at work.
With at most 10% in any stock, the best you can do is spread the money over the 10 best stocks, and that portfolio
fits under a fairly low risk limit. Past the dotted line, the risk limit is no longer the constraint that binds; your
rules are. The right panel zooms in on the orange line so you can see its own short climb before it flattens.

**The star is the top-left corner**, the spot to pick: the most return for the risk, just before extra risk stops
paying. The code finds it on its own. It scales both axes from 0 to 1, draws a straight line from the first point to
the last, and takes the point farthest above that line. You can always pick a different spot; the Bonus at the end
shows you how to check any risk limit you like.

### The allocation behind each point on the frontier

Every dot on a frontier is a whole portfolio. This picture stacks the shares, so you can see how the money moves from
stock to stock as you allow more risk. Read it left to right: each color is one stock, and its height is that
stock's share of the money at that risk limit.

In [ ]:
def allocation_table(weights_by_risk):
    # one row per risk limit, one column per stock that ever gets money
    table = pd.DataFrame(weights_by_risk).T
    keep = []
    for s in table.columns:
        if table[s].max() > 0.001:
            keep.append(s)
    table = table[keep].clip(lower=0)
    return table


free_alloc = allocation_table(free_weights)
ruled_alloc = allocation_table(ruled_weights)

# the same color for a stock in both panels
all_held = list(free_alloc.columns)
for s in ruled_alloc.columns:
    if s not in all_held:
        all_held.append(s)
colors = {}
for k in range(len(all_held)):
    colors[all_held[k]] = cm.tab20(k % 20)

fig, panels = subplots(1, 2, figsize=(15, 5.5), sharey=True)

free_colors = []
for s in free_alloc.columns:
    free_colors.append(colors[s])
panels[0].stackplot(free_alloc.index, free_alloc.T.values, colors=free_colors)
panels[0].set_title('No rules')
panels[0].set_xlabel('Risk limit (variance of the monthly return)')
panels[0].set_ylabel('Share of the money')

ruled_colors = []
for s in ruled_alloc.columns:
    ruled_colors.append(colors[s])
panels[1].stackplot(ruled_alloc.index, ruled_alloc.T.values, colors=ruled_colors)
panels[1].axvline(corner_risk, color='black', linestyle='--', linewidth=2)
panels[1].set_title('With your rules (zoomed range; dashed line = top-left corner)')
panels[1].set_xlabel('Risk limit (variance of the monthly return)')

handles = []
for s in all_held:
    handles.append(Rectangle((0, 0), 1, 1, color=colors[s]))
panels[1].legend(handles, all_held, loc='center left', bbox_to_anchor=(1.01, 0.5), fontsize=8)
show()

print("your portfolio at the top-left corner (percent of the money):")
corner_mix = ruled_alloc.loc[corner_risk]
corner_mix = corner_mix[corner_mix > 0.001].sort_values(ascending=False)
print((100 * corner_mix).round(1).to_string())

**Read the allocation plots.** With no rules, the money collapses into just a few stocks as you allow more risk:
the bands narrow until one or two colors fill the top. With your rules no band can grow past 10% or shrink below 2%,
so the stack stays spread out and, past the flat point, stops changing at all.

# Part 4 · Backtest: would your rules have worked last year?

Everything so far used all five years to pick the portfolio, so it has already "seen" every month it is judged on.
A fair test hides some months from the model. Here we **pretend it is 12 months ago**:

1. Fit both models (no rules, and with your rules) on the first four years only, and find the top-left corner of the
   rules frontier the same way as in Part 3.
2. Put \$10,000 into each portfolio at the start of the hidden year and hold it, month by month.
3. Compare them with two simple baselines: an equal split of your stocks, and the S&P 500 (the SPY fund).

The model never sees the year it is tested on. This is the same question your final report asks, on months the
model has not seen.

In [ ]:
# pretend it is 12 months ago: fit on the first four years, test on the last 12 months
test_months = 12
train = returns.iloc[:-test_months]
test = returns.iloc[-test_months:]
print("fit on   ", train.index[0].date(), "to", train.index[-1].date(), "(%d months)" % len(train))
print("test on  ", test.index[0].date(), "to", test.index[-1].date(), "(%d months, hidden from the model)" % len(test))

# solve_portfolio reads df_return and df_cov, so point them at the training months for now
full_return = df_return
full_cov = df_cov
df_return = train.mean()
df_cov = train.cov()

# the training frontier with your rules, and its top-left corner (the same method as Part 3)
train_even_risk = 0
for i in stocks:
    for j in stocks:
        train_even_risk += df_cov.loc[i, j] / len(stocks) ** 2
train_best = solve_portfolio(df_cov.values.diagonal().max(), use_rules=True)
train_zoom = {}
for r in np.linspace(train_even_risk * 0.5, train_best['risk'] * 1.3, 12):
    b = solve_portfolio(r, use_rules=True)
    if b is not None:
        train_zoom[r] = b['return']
backtest_limit = find_corner(train_zoom)

backtest_free = solve_portfolio(backtest_limit, use_rules=False)
backtest_ruled = solve_portfolio(backtest_limit, use_rules=True)

# put the full five years back for everything after this cell
df_return = full_return
df_cov = full_cov

print("risk limit at the top-left corner of the training frontier: %.4f" % backtest_limit)
if backtest_free is None or backtest_ruled is None:
    print("A model could not be solved at that risk limit. Try a different test_months, or loosen a rule.")

In [ ]:
# hold each portfolio through the hidden year, starting with $10,000
start_money = 10000
growth = (1 + test).cumprod()


def hold(weights):
    value = 0
    for s in stocks:
        value = value + start_money * weights[s] * growth[s]
    # add the starting point, the month before the test year
    start_point = pd.Series([start_money], index=[train.index[-1]])
    return pd.concat([start_point, value])


equal_weights = {}
for s in stocks:
    equal_weights[s] = 1 / len(stocks)

paths = {}
paths['no rules'] = hold(backtest_free['weights'])
paths['with your rules'] = hold(backtest_ruled['weights'])
paths['equal split of your stocks'] = hold(equal_weights)

# the S&P 500, if Yahoo gives it to us
try:
    spy = yf.download('SPY', start=start, end=end, interval='1mo', auto_adjust=True, progress=False)['Close'].squeeze()
    spy_returns = spy.pct_change().reindex(test.index)
    if spy_returns.isna().sum() == 0:
        spy_path = start_money * (1 + spy_returns).cumprod()
        paths['S&P 500 (SPY)'] = pd.concat([pd.Series([start_money], index=[train.index[-1]]), spy_path])
    else:
        print("SPY months did not line up, so the S&P 500 line is left out")
except Exception as problem:
    print("SPY did not download, so the S&P 500 line is left out")

line_colors = {'no rules': 'gray', 'with your rules': 'tab:orange', 'equal split of your stocks': 'tab:blue',
               'S&P 500 (SPY)': 'black'}
figure(figsize=(11, 5))
for name in paths:
    plot(paths[name].index, paths[name].values, marker='o', color=line_colors[name], label=name)
axhline(start_money, color='lightgray', linestyle='--')
ylabel('Value of $10,000')
title('The hidden year: $10,000 in each portfolio')
legend()
show()

# one row per portfolio
promised = {'no rules': backtest_free['return'], 'with your rules': backtest_ruled['return']}
rows = {}
for name in paths:
    path = paths[name]
    monthly = path.pct_change().dropna()
    worst_drop = (1 - path / path.cummax()).max()
    row = {}
    row['ending value ($)'] = round(path.iloc[-1])
    row['total return (%)'] = round(100 * (path.iloc[-1] / start_money - 1), 1)
    row['average month (%)'] = round(100 * monthly.mean(), 2)
    row['promised average month (%)'] = '-'
    if name in promised:
        row['promised average month (%)'] = round(100 * promised[name], 2)
    row['worst drop (%)'] = round(100 * worst_drop, 1)
    rows[name] = row
backtest_table = pd.DataFrame(rows).T
backtest_table

**Read the backtest.** Compare the "promised" average month (what the model expected from the four training years)
with the average month it actually earned in the hidden year. It is almost always lower: the model picks the stocks
whose past looks best, and some of that past was luck. Then look at how the rules did against no rules. Did spreading
the money out cost you return, protect you from a bad drop, or both? One year is a small sample, so treat this as a
first look, not a verdict. The live leaderboard is the real test.

### Your turn

Answer in a text cell:

1. Which stocks did you download, and did any fail or get dropped? (Part 1)
2. Which stock had the best average month, and how much of the money did the model put in it **without** rules?
3. What did the rules cost at the Step 6 risk limit, in percentage points a month and in dollars on \$10,000?
4. Change one rule (for example `max_share` to 0.20, or `max_stocks` to 12) and re-run Steps 5 and 6. What changed?
5. Look at Part 3. Where does your orange line go flat, and which rule is holding it back? Try `max_share = 0.20` and see what happens to the flat point.
6. Look at Part 4. In the hidden year, did your rules beat no rules? Did either beat the S&P 500? How far was each model's promise from what it earned?

## Where this is going: the final project

Everything tonight ran once, by hand, on five years of history. In the final project the same code will run **on its
own every month in the cloud**: download the newest prices, re-solve your model with your rules, and record what your
orange-line portfolio would have held, so you can check how it actually did on months the model has not seen yet. Dr. Dave will show you how
that is set up; the notebook you finished tonight is the engine.

# Bonus · Why does the orange line go flat? Compare each side of the constraints

Every constraint has a **left-hand side** (what the portfolio actually comes to) and a **right-hand side** (the limit).
The gap between them is the **slack**.

- **Slack = 0:** the constraint is **binding**. It is what holds the answer back. Loosen it, and the return can go up.
- **Slack > 0:** the constraint has room to spare. Loosening it changes nothing.

Below we check every constraint at **the top-left corner** from Part 3 (change `chosen_limit` to check any spot you
like), and then far **past** the flat point, for comparison.

In [ ]:
def check_constraints(name, limit):
    result = solve_portfolio(limit, use_rules=True)
    print(name, "- risk limit %.6f, best return %.2f%% a month" % (limit, 100 * result['return']))

    held = []
    for s in stocks:
        if result['weights'][s] > 0.001:
            held.append(s)

    # the main constraints: left-hand side vs right-hand side
    total = 0
    for s in stocks:
        total += result['weights'][s]
    rows = {}
    rows['invest all the money (sum of x = 1)'] = [total, 1]
    rows['keep risk under the limit (variance <= limit)'] = [result['risk'], limit]
    for sector in rules['must_have_sectors']:
        count = 0
        for s in held:
            if my_stocks.get(s) == sector:
                count += 1
        rows['at least one %s stock (number held >= 1)' % sector] = [count, 1]
    if rules['max_stocks'] is not None:
        rows['at most %d stocks (number held <= %d)' % (rules['max_stocks'], rules['max_stocks'])] = [len(held), rules['max_stocks']]

    table = pd.DataFrame(rows, index=['left-hand side', 'right-hand side']).T
    table['slack'] = (table['right-hand side'] - table['left-hand side']).abs()
    table['binding?'] = 'no'
    table.loc[table['slack'] < 1e-6, 'binding?'] = 'YES'
    display(table.round(6))

    # the linking constraints, one pair per stock held: 2% floor <= x <= 10% cap
    stock_rows = {}
    for s in held:
        x = result['weights'][s]
        where = 'in between'
        if rules['max_share'] is not None and x >= rules['max_share'] - 0.001:
            where = 'AT the cap (binding)'
        if rules['min_share'] is not None and x <= rules['min_share'] + 0.001:
            where = 'AT the floor (binding)'
        stock_rows[s] = [round(x, 4), where]
    display(pd.DataFrame(stock_rows, index=['share x', 'where it sits']).T.sort_values('share x', ascending=False))
    print()


# the spot to analyze: the top-left corner from Part 3, unless you type your own risk limit
chosen_limit = corner_risk
# chosen_limit = 0.0015

check_constraints("YOUR CHOSEN SPOT (the top-left corner unless you changed it)", chosen_limit)
check_constraints("FAR PAST the flat point, for comparison", top)

**What to notice.** On the climbing part of the orange line, the risk constraint is binding (slack 0): allow a little
more risk and the return goes up. Far past the flat point, the risk constraint has
lots of slack, and the stocks are stacked at exactly 10%: now the **cap** is the binding constraint. That is why the
line goes flat. Raising the risk limit loosens a constraint that was not holding anything back. To move the flat line
up, you would have to loosen the cap itself (try `max_share = 0.20` in Step 5).